In [ ]:
# 학습된 모델을 테스트 해볼 수 있는 코드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U qwen-tts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
  Created wheel for sox: filename=sox-1.5.0-py3-none-any.whl size=40036 sha256=70636d8195a1945f204dc977c956bab466dd16da505221f1ba6f72fc9b225dc0
  Stored in directory: /root/.cache/pip/wheels/8c/c7/e7/baea1f7e79b9eb53addc81cc9b827424f4a7d8c9cc18c03659
Successfully built sox
  Attempting uninstall: huggingface_hub
    

In [ ]:
!apt-get update -qq
!apt-get install -y sox libsox-fmt-all

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3 libsox-fmt-oss
  libsox-fmt-pulse libsox3 libwavpack1
Suggested packages:
  libaudio2 libsndio6.1
The following NEW packages will be installed:
  libao-common libao4 libid3tag0 libmad0 libopencore-amrnb0 libopencore-amrwb0
  libsox-fmt-all libsox-fmt-alsa libsox-fmt-ao libsox-fmt-base libsox-fmt-mp3
  libsox-fmt-oss libsox-fmt-pulse libsox3 libwavpack1 sox
0 upgraded, 16 newly installed, 0 to remove and 88 not upgraded.
Need to get 800 kB of archives.
After this operation, 2,533 kB of additional disk space will be

In [ ]:
# 하나의 체크포인트(모델)를 선택해서 다양하게 테스트
# 문장, 감정을 파라미터로 입력 가능
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

import torch
import soundfile as sf
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio, display

# 통제변수 speaker
speaker = "jhc"
ckpt_root = f"/content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/{speaker}"
selected_epoch_num = None  # None이면 최신

if selected_epoch_num is None:
    epochs = []
    for name in os.listdir(ckpt_root):
        if name.startswith("checkpoint-epoch-"):
            num = int(name.split("-")[-1])
            epochs.append(num)
    selected_epoch_num = max(epochs)

print("선택 epoch:", selected_epoch_num)

target_ckpt = os.path.join(
    ckpt_root, f"checkpoint-epoch-{selected_epoch_num}"
)

if not os.path.exists(target_ckpt):
    raise FileNotFoundError(f"해당 epoch 없음: {target_ckpt}")

if not os.path.exists(os.path.join(target_ckpt, "model.safetensors")):
    raise RuntimeError(f"가중치 없음: {target_ckpt}")

print("선택된 체크포인트:", target_ckpt)

tts = Qwen3TTSModel.from_pretrained(
    target_ckpt,
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

texts = [
    "안녕하세요, 오늘 하루도 차분하게 시작해 보겠습니다.",
    "오늘 점심은 볶음밥에 계란을 먹어야겠어.",
    "지금 시간은 오후 세 시입니다.",
    "와, 생각보다 훨씬 자연스럽게 들리네요!",
    "괜찮아요, 천천히 다시 해보면 됩니다.",
    "야 너 왜 그렇게 사냐",
    "개 짖는 소리 좀 안나게 해라",
    "This is my TTS, bro",
    "1,2,3,4,5",
    "010-1234-5677",
]

# instruct : 감정 조절 부분 ex) happy, sad, angry ...
wavs, sr = tts.generate_custom_voice(
    text=texts,
    speaker=speaker,
    instruct="very angry"
)

for i, wav in enumerate(wavs):
    out_path = f"/content/{speaker}_epoch{selected_epoch_num}_{i+1}.wav"
    sf.write(out_path, wav, sr)

    print(f"\n===== 문장 {i+1} =====")
    print(texts[i])
    display(Audio(out_path))

선택 epoch: 3
선택된 체크포인트: /content/drive/MyDrive/비트메이트_TP01/qwen3_ft_output/jjy/checkpoint-epoch-3


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



===== 문장 1 =====
안녕하세요, 오늘 하루도 차분하게 시작해 보겠습니다.



===== 문장 2 =====
오늘 점심은 볶음밥에 계란을 먹어야겠어.



===== 문장 3 =====
지금 시간은 오후 세 시입니다.


...


===== 문장 8 =====
1,2,3,4,5



===== 문장 9 =====
010-1234-5677



===== 문장 10 =====
아 그거 내 두쫀쿠라고



===== 문장 11 =====
서폿 럭스 개빡치네



===== 문장 12 =====
개 짖는 소리 좀 안나게 해라
